# Parser sintático e avaliação de lógica proposicional (Cópia_de_RA1_Parser_Logica_Proposicional.ipynb)

- **Repositório:** `analise-de-dados-fundamentos`  
- **Tecnologias:** Python  
- **Arquivo Original:** `Cópia_de_RA1_Parser_Logica_Proposicional.ipynb`  

---


## Célula de leitura do arquivo

In [34]:
'''
  def ler_arquivo(nome_arquivo):
      try:
          with open(nome_arquivo, "r") as f:
              linhas = f.readlines()
          return linhas
      except FileNotFoundError:
          print("Arquivo de entrada não encontrado!")
          return []


  linhas_arquivo = ler_arquivo("testes.txt")
'''

'\n  def ler_arquivo(nome_arquivo):\n      try:\n          with open(nome_arquivo, "r") as f:\n              linhas = f.readlines()\n          return linhas\n      except FileNotFoundError:\n          print("Arquivo de entrada não encontrado!")\n          return []\n\n\n  linhas_arquivo = ler_arquivo("testes.txt")\n'

## Conjunto de Regras

In [35]:
regras = {
    "FORMULA": "CONSTANTE | PROPOSICAO | FORMULAUNARIA | FORMULABINARIA",
    "CONSTANTE": "true | false",
    "PROPOSICAO": "[0-9][0-9a-z]*",
    "FORMULAUNARIA": "( OPERATORUNARIO FORMULA )",
    "FORMULABINARIA": "( OPERATORBINARIO FORMULA FORMULA )",
    "OPERATORUNARIO": "\\neg",
    "OPERATORBINARIO": "\\wedge | \\vee | \\rightarrow | \\leftrightarrow",
    "ABREPAREN": "(",
    "FECHAPAREN": ")"
}



In [36]:
# Extrai operadores e constantes
operadores_binarios = [op.strip() for op in regras["OPERATORBINARIO"].split("|")]
operadores_unarios = [regras["OPERATORUNARIO"]]
constantes = [c.strip() for c in regras["CONSTANTE"].split("|")]

# Verifica se é uma proposição válida
def eh_proposicao(token):
    if not token or not token[0].isdigit():
        return False
    for c in token[1:]:
        if not (c.isdigit() or 'a' <= c <= 'z'):
            return False
    return True

# Verifica se é constante
def eh_constante(token):
    return token in constantes

# Função principal que analisa a fórmula
def validar_formula(tokens):
    def parse(index):
        if index >= len(tokens):
            return -1

        token = tokens[index]

        if token == regras["ABREPAREN"]:
            if index + 1 < len(tokens):
                prox = tokens[index + 1]

                # FORMULA UNARIA
                if prox in operadores_unarios:
                    prox_index = parse(index + 2)
                    if prox_index != -1 and prox_index < len(tokens) and tokens[prox_index] == regras["FECHAPAREN"]:
                        return prox_index + 1
                    else:
                        return -1

                # FORMULA BINARIA
                elif prox in operadores_binarios:
                    esq_index = parse(index + 2)
                    if esq_index == -1:
                        return -1
                    dir_index = parse(esq_index)
                    if dir_index != -1 and dir_index < len(tokens) and tokens[dir_index] == regras["FECHAPAREN"]:
                        return dir_index + 1
                    else:
                        return -1

        elif eh_constante(token) or eh_proposicao(token):
            return index + 1

        return -1

    resultado = parse(0)
    return resultado == len(tokens)

In [37]:
class StateMachine:
    def __init__(self):
        self.state = 'idle'

        self.transitions = {
            'start': {'CONSTANTE': 'formula_simples_correta',
                      'PROPOSICAO': 'formula_simples_correta',
                      'ABREPAREN': 'abre_parenteses'},

            'formula_simples_correta': {' ': 'formula_simples_correta'},

            'abre_parenteses': {' ': 'abre_parenteses',
                                'ABREPAREN': 'abre_parenteses',
                                'OPERADORUNARIO': 'operador_unario',
                                'OPERADORBINARIO': 'operador_binario',},

            'operador_unario': {' ': 'operador_unario',
                                'CONSTANTE': 'termo_unario',
                                'PROPOSICAO': 'termo_unario',
                                'ABREPAREN': 'abre_parenteses'},

            'termo_unario': {' ': 'termo_unario',
                             'FECHAPAREN': 'fecha_parenteses'},

            'operador_binario': {' ': 'operador_binario',
                                 'ABREPAREN': 'abre_parenteses',
                                 'CONSTANTE': 'termo_binario_nao_final',
                                 'PROPOSICAO': 'termo_binario_nao_final' },

            'termo_binario_nao_final': {' ': 'termo_binario_nao_final',
                                  'CONSTANTE': 'termo_binario_final',
                                  'PROPOSICAO': 'termo_binario_final',
                                  'ABREPAREN': 'abre_parenteses'},


            'termo_binario_final': {' ': 'termo_binario_final',
                                        'FECHAPAREN': 'fecha_parenteses'},

            'fecha_parenteses': {' ': 'fecha_parenteses',
                                 'ABREPAREN': 'abre_parenteses',
                                 'FECHAPAREN' : 'fecha_parenteses',
                                 'CONSTANTE' : 'termo_binario_final',
                                 'PROPOSICAO': 'termo_binario_final'}
        }

        self.final_states = {'formula_simples_correta', 'fecha_parenteses'}

    def trigger(self, event):
        if event in self.transitions[self.state]:
            new_state = self.transitions[self.state][event]
            self.state = new_state

    def process(self, tokens):
      self.state = 'start'

      tokens_converted = [] # convertendo cada token para simplificar o processo da maquina de estados
      for token in tokens:
        if eh_constante(token):
          tokens_converted.append('CONSTANTE')

        elif token == '(':
          tokens_converted.append('ABREPAREN')
        elif token == ')':
          tokens_converted.append('FECHAPAREN')

        elif token == '\\neg':
          tokens_converted.append('OPERADORUNARIO')

        elif token == '\\wedge' or token == "\\vee" or token == "\\rightarrow" or token == "\\leftrightarrow":
          tokens_converted.append('OPERADORBINARIO')

        elif eh_proposicao(token):
          tokens_converted.append('PROPOSICAO')

        else:
          tokens_converted.append(token)

      for token in tokens_converted:
          if token not in self.transitions.get(self.state, {}):
              return False
          self.trigger(token)

      if(self.state in self.final_states):
        return tokens

In [39]:
# Lê o arquivo testes.txt
caminho_arquivo = "/content/testes.txt"

with open(caminho_arquivo, "r", encoding="utf-8") as f:
    linhas = f.readlines()

qtd = int(linhas[0].strip())
expressoes = [linha.strip() for linha in linhas[1:]]

# Valida e imprime o resultado de cada expressão
for i, expr in enumerate(expressoes, 1):
    tokens = expr.replace("(", " ( ").replace(")", " ) ").split()

    fsm = StateMachine()
    result = fsm.process(tokens)

    print(f"Expressão {i}: {expr}")

    if not result:
        print("Resultado: inválida\n")
    else:
        if validar_formula(result):
            print("Resultado: válida\n")
        else:
            print("Resultado: inválida\n")


Expressão 1: 15
Resultado: válida

Expressão 2: true
Resultado: válida

Expressão 3: false
Resultado: válida

Expressão 4: ( \neg 0a )
Resultado: válida

Expressão 5: ( \wedge true ( \neg false ) )
Resultado: válida

Expressão 6: ( \vee ( \neg 2x ) ( \vee 3y 4z ) )
Resultado: válida

Expressão 7: ( \leftrightarrow ( \neg 1a ) ( \wedge 2b ( \neg false ) ) )
Resultado: válida

Expressão 8: ( \wedge 1a 2b 3c )
Resultado: inválida

Expressão 9: ( \wedge ( \neg ) 2a )
Resultado: inválida

Expressão 10: ( \vee ( \wedge 1a ) 3b )
Resultado: inválida

Expressão 11: ( \vee 1a )
Resultado: inválida

Expressão 12: ( \wedge ( \vee 1a 2b ) ( \neg ( \rightarrow 3c ( \vee 4d 5e ) ) ) )
Resultado: válida

Expressão 13: ( \neg ( \neg ( \neg 6x ) ) )
Resultado: válida

Expressão 14: ( \neg ( \rightarrow true ) )
Resultado: inválida

Expressão 15: ( \leftrightarrow 1a 2b )
Resultado: válida

Expressão 16: ( \neg 123abc )
Resultado: válida

